# Pac-Man correction round — native-v2 → native-v3

Continue from your **completed `kev-4b-pacman-native-v2` LoRA and pointer head**. Keep original base matrices frozen. This is a new supervised stage with a fresh optimizer/schedule, not a resume of v2's completed 762 updates. V3 saves to its own output and recovery directory. Stages 1–4 and v2 need no retraining.

The v2 audit found substantially better food routing but zero maze clears, nearly suppressed reversals and weak recovery after death. This follow-up uses fresh learner trajectories, teacher corrections and independently replayed full teacher recoveries. It keeps Kev's cross-entropy loss and all-module rank-16 LoRA/head architecture. This is a DAgger-style correction mixture, not a reproduction of CLM or the original DAgger schedule. [DAgger paper](https://arxiv.org/abs/1011.0686).

**Data generation is instructor prework.** Collecting learner games uses the v2 model on GPU. Hundreds of complete teacher continuation simulations then run on CPU and can take hours; progress prints every 15 seconds and after each attempt. Pause the game and keep the runtime connected. Teacher successes and failures are retained, and coverage floors must pass before a training dataset is admitted. CPU checks do not establish v3 GPU fit, training duration or improved play. The main lab still has one assessed CP1.

Run setup/storage, load v2, collect fresh trajectories, verify/build labels, train v3, then compare full games and play. On a restored completed v3 run, skip collection/build/training and go straight to comparison or interactive play.

## Prepare the optimized runtime and restore Drive checkpoints

Use the same RTX PRO 6000 Blackwell runtime as the main lab. The following cells reuse its pinned dependencies and optimized FLA/convolution/fused-AdamW stack. Drive backup is enabled by default. Storage automatically restores available v2/v3 checkpoints and recovery; v3 backups also contain its exact training input, development labels, dataset receipt and native replay bundle. The adapter-loading cell below also restores teacher-generation prework into an absent collection directory.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, json, os, sys
TRAINING_PROFILE = 'memory_safe'  # Later stages; Stage 1 explicitly selects 4 x 2 below
LAB_DIR = Path.cwd() / 'pacman-kev-lab'
LAB_DIR.mkdir(exist_ok=True)
COURSE_REVISION = 'bbc174c8825b8e23a20ef62856f96cd303db35d2'
FILES = {'api_client.py': 'd24f52b4bfb0c85fa1f4e67ab295eaa08e0f5654e71e6f0e0b030d7fb5347de6', 'pacman_lab.py': 'ebfa966b1a8f4bbaf90c98eacb11e77f2c18ec04158bee6744d1aee960307bc8', 'cloud_runtime.py': '9ad546991e4542d5d7d540b67d49218b0d3f6a656c209385a5111c0ff8deda52', 'training_monitor.py': '816f46da63ab7e936d55e85425ab9213145f2276fe6f2a260248307b45fbe681', 'lora_recovery.py': 'd01bc4f336b17948dc89fa8e931761c766fed979349a310d19f23e5ce0d6e274', 'checkpoint_backup.py': 'a59c305ae279170a069b8194ff176800ca9702a3651dcb92442e2631ba18db7c', 'optimized_training.py': '5c33fac17ec6ad6439193737aea650f85cdfb4445a0d346bb5bed10b858fcdfa', 'training-kernels.json': '24dff024433f61c4a571f16326f4f16c5666297e6514089e66e721b39765e56c', 'training_stages.py': '96fdd59285b8872b946523a88d95393decb73536322ca35e8f67f79cb2a7ef5c', 'training-stages.json': '9be232b730a9550a01011091b1abf3d79eaba3de8db5bdd455783fe5ae0e8ad0', 'games/arcade-engine.js': 'df5cf83c69810c641b65ccb533dd5da5b0ca6aca619db32090ad932fd8cd7e16', 'games/arcade-worker.js': 'acd16571bd330e2dc8033ab7c59977f192ae634d1eec4291db7e16a09b730daa', 'games/arcade-browser.js': 'e6f7fcba0e8d5c79573fe461e89efb428584acea594e9ec1e351e5d2964a0395', 'games/arcade-shell.html': 'b2606cdd679242c434570fd3850618041f79a8590f3b813106536c4d78341a0c', 'vendor/arcade-pacman/source.json': 'a0f21cf6009b7f7ca195fb8ce5f0730609767e7ced1c9b8cf29152f5bbe52545', 'vendor/arcade-pacman/source.zip': 'fd9ae0cbeebdfd798c609464e884f6e9c9b59673bd50083fbd0e32ea6c26a6e8', 'planner_data.py': '9b325f71bdc08419014b6e69b1541868d483c20efaaf79964892ac3887e65ca4', 'games/arcade-planner.js': 'f505a2b131465f0f170e16731863efe870e8549b50239f41d5a230dd57b7ff57', 'data/pacman-planner-v1.zip': 'a6f7e5282e888ede6a24f28cc7dffdb7d3cf51776ec5c9d6adefcf6e15e9e6a1', 'data/pacman-planner-v1-manifest.json': '3c22ede98985264b46410dc14026eaa22301b79a3f9061cf5063c739fc39b6e2', 'data/pacman-planner-v1-quality.json': 'db6ae0d8b730c7582f3f76e0c0177061d5449cfd55bc87ae3dd4c66277d54fc0', 'gameplay_benchmark.py': '62f205f9db11c7e067dbd07f609d0835fe92e1d2de3627aee2f7b641d2ed2008', 'benchmark-spec.json': '6be3f95e110422defe40c5c9007617f9e09f8c0e5c86a16b9107a73e52512cdc', 'games/benchmark-hooks.js': '9087a8ee70d8bb25c71a84abe8f614a4ea99f897ba345e36f4a70f520dca4069', 'games/benchmark-worker.cjs': '1d30685f16f5c55041fc5c50156da1119bd96c2ef382e09d4c7048939f7b1df7', 'games/arcade-teacher.js': '66d98198b77ad5364688006ed132804fae1ccc395fb4a4536980abbdafbe2f3b', 'teacher_validation.py': '0dfd7e7965a4837a7a989831802253d7f57a830283b241bf659ade7c876e8084', 'teacher-validation-spec.json': 'f5a57378e7020e543b14d4ab8ebaa0628097393307c2c493b96c0eb26d736780', 'teacher_data.py': '095a75cd23d2386987ce40d250b0fbd1979a01a97265ebc3a397aec749a847f0', 'evaluation/teacher-qualification.json': 'bd848daed094f60e3c84dc267f2569f644bf8ce04c324d79942bf0dd821dbce5', 'evaluation/teacher-qualification-replays.zip': 'bb21bcbf33ab5332c35a4f6022f8e4a1a4f70ef1546c0e5c3218142fc8d520c2', 'data/pacman-native-v2.zip': '8327bd809c86172283366cf018321e59ba91ceafc2d798555b28cb0aab6e7651', 'data/pacman-native-v2-manifest.json': '8cc0a7b26b010ff4bb1866e37f5bb3f5841c3c4a6f81d932d9ae52ba4971ae0f', 'data/pacman-native-v2-replays.zip': '6cfb6378951af88039f9503c13964a47cf83f7adaeb3d32245e69342b9652545', 'data/pacman-native-v2-generator.py': '31b7d5dcd4f750db0698f22535f13667f79e77e97cee525f373b8f99a9326be9', 'v3_data.py': '5220adf63df9326ac05c0581e7bffbb682ca1b4df4709152dd2a48e276c8d2d1', 'v4_data.py': '8a35c4f907d1391471b23ce11f6006141f931e6b00a00b408046e2a2ed0d6063', 'teacher_offline_data.py': 'e35f4ba38b45807637c8a4502326a31656c89071ee74fb2bf0a2d2f8e936ef48', 'scenario_catalog.py': '1962c676744570214a6d369ecd8f10a9343db16e039a85e2f73468c04f3570b4', 'collection_backup.py': '1f5b7b8f8412786ba504ed640720c931a44381e61b9557b457423d4bb125c625', 'hard_case_evaluation.py': '6827a4fdc52980f9c1b1240a1d1879e3cace8ea0ef8f6b23c8edc0d7a583c0b7', 'evaluation/v3-analysis-2026-10-07/teacher-reference.json': 'c3c06bb5a5de1cca6559da62206a1314aefa17c544c97bd7171b22903014c888'}
for name, expected_sha256 in FILES.items():
    target = LAB_DIR / name
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == expected_sha256:
        continue
    url = f'https://raw.githubusercontent.com/yxc20089/QPlusLearning/{COURSE_REVISION}/labs/lab-01-kev-pacman/{name}'
    with urlopen(url, timeout=60) as response:
        content = response.read()
    assert hashlib.sha256(content).hexdigest() == expected_sha256, f'Unexpected source: {name}'
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
    print('Downloaded verified helper:', name, flush=True)
sys.path.insert(0, str(LAB_DIR))
# Refresh helpers when upgrading an existing notebook; keep checkpoints/logs.
if 'runtime' in globals():
    runtime.stop()
import importlib
importlib.invalidate_caches()
for name in ['cloud_runtime', 'training_monitor', 'lora_recovery', 'checkpoint_backup', 'optimized_training', 'training_stages', 'gameplay_benchmark', 'teacher_validation', 'teacher_data', 'planner_data', 'pacman_lab', 'api_client', 'v3_data', 'v4_data', 'collection_backup', 'hard_case_evaluation']:
    sys.modules.pop(name, None)
from cloud_runtime import CloudRuntime
from lora_recovery import latest_snapshot
from checkpoint_backup import new_run_directory
from training_stages import specifications, inspect_checkpoint, restore_checkpoint, backup_checkpoint
from pacman_lab import *
from teacher_data import prepare_dataset, PREFIX, RECIPE
from gameplay_benchmark import benchmark_gameplay, paired_gameplay
from api_client import call, distribution
ensure_node(LAB_DIR)
runtime = CloudRuntime(LAB_DIR, training_profile=TRAINING_PROFILE)
CHECKPOINT_ROOT = LAB_DIR / 'checkpoints'
INITIAL = CHECKPOINT_ROOT / 'kev-4b-initial'
DATES = CHECKPOINT_ROOT / 'kev-4b-dates'
DOCUMENTS = CHECKPOINT_ROOT / 'kev-4b-documents'
SKILLS = CHECKPOINT_ROOT / 'kev-4b-skills'
STAGES = specifications()
manifest = None  # Loaded with its teacher qualification receipt in CP1
GAME = notebook_game(action_version=2)
STAGE_OWNERS = {}
print('Prepared the native arcade engine, four ghosts and full-game benchmark. CP1 verifies qualified demonstrations.')

In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'tensorboard==2.20.0'])
LOG_DIR = LAB_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)
SHOW_TENSORBOARD = True
if SHOW_TENSORBOARD:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic('load_ext', 'tensorboard')
        ip.run_line_magic('tensorboard', f'--logdir "{LOG_DIR}" --reload_interval 5')
    else:
        print('Outside a notebook, run: tensorboard --logdir', LOG_DIR)

In [ ]:
runtime.setup()

In [ ]:
RUN_TRAINING_PREFLIGHT = False  # Optional extra model checks; optimized bindings remain required
audit = runtime.prepare_training(run_training_preflight=RUN_TRAINING_PREFLIGHT)
print('Training suite records:', audit['suite_records'])
print('Trainable parameters:', sum(audit['trainable_parameters'].values()))
print('Optimized training verification:', json.dumps(runtime.training_preflight, indent=2))

In [ ]:
SAVE_TO_DRIVE = True  # True enables automatic backup and restore; no manual file copying
RESTORE_RECOVERY_ARCHIVE = None  # ZIP from the inspection/export cell; restore into an absent root
CHECKPOINT_ROOT = LAB_DIR / 'checkpoints'
DRIVE_BACKUP_ROOT = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP_ROOT = Path('/content/drive/MyDrive/QPlusLearning/lab-01-kev-pacman/backups')
if RESTORE_RECOVERY_ARCHIVE is not None:
    restore_checkpoint(RESTORE_RECOVERY_ARCHIVE, CHECKPOINT_ROOT)
else:
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print('Automatic Drive backup:', json.dumps(runtime.configure_backup(DRIVE_BACKUP_ROOT, CHECKPOINT_ROOT), indent=2))
print('Later-stage training profile:', runtime.training_profile, 'overrides:', runtime.selected_recipe({}))
print('Stage 1 selects batch 4 x accumulation 2 / row_budget 0 in its own training cell.')
print('Checkpoints and recovery:', CHECKPOINT_ROOT)

## Load the completed native-v2 adapter

The default path is restored by the storage cell from `MyDrive/QPlusLearning/lab-01-kev-pacman/backups/kev-4b-pacman-native-v2`. The active model identity must identify this completed adapter. Keep v2 intact: it supplies both the learner policy for data collection and the before-model for comparison.

In [ ]:
from v3_data import collect, build, validate_dataset, behavior_summary, PREFIX as V3_PREFIX, COUNTS as V3_COUNTS, MINIMUMS
from training_stages import checkpoint_fingerprint
from checkpoint_backup import restore_teacher_collection
V2 = CHECKPOINT_ROOT / 'kev-4b-pacman-native-v2'
V3 = CHECKPOINT_ROOT / 'kev-4b-pacman-native-v3'
V3_DATA = LAB_DIR / 'data'
V3_TRAINING = V3_DATA / f'{V3_PREFIX}-train.jsonl'
QUALIFICATION = LAB_DIR / 'evaluation/teacher-qualification.json'
COLLECTION_DIR = LAB_DIR / 'results/v3-collection'
TEACHER_BACKUP_ROOT = (runtime.backup_root.parent / 'prework-backups/v3-collection'
                       if runtime.backup_root is not None else None)
if TEACHER_BACKUP_ROOT is not None:
    restore_teacher_collection(COLLECTION_DIR, TEACHER_BACKUP_ROOT)
print('Teacher prework Drive backup:', TEACHER_BACKUP_ROOT)
if not (V2 / 'run-evidence.json').is_file():
    raise RuntimeError(f'Restore the completed native-v2 checkpoint to {V2}. This round does not start from Skills.')
runtime.start(V2)
V2_IDENTITY = runtime.active_model_info()
print('Correction learner / warm-start parent:', json.dumps(V2_IDENTITY, indent=2))
print('New v3 output:', V3)
print('Training input:', V3_TRAINING)
print('Coverage floors:', json.dumps(MINIMUMS, indent=2))

## Collect fresh learner trajectories — prework

V2 plays 16 training and eight development games at levels 1, 2, 3 and 5, using fresh, disjoint seeds. The original 20 benchmark starts, old data seeds, and teacher development/qualification seeds are excluded. No teacher or safety filter replaces a learner choice. Full state/action/probability traces and native diagnostics are retained under `results/v3-collection/`. Completed partitions are reused when this cell is rerun with the same adapter and seeds.

Skip this cell when the verified v3 dataset or a completed v3 checkpoint is already restored. Changing the learner or seeds requires a new collection directory.

In [ ]:
# Fresh native-v2 rollouts for correction data; not the reserved benchmark
if (V3_DATA / f'{V3_PREFIX}-manifest.json').is_file() or (V3 / 'run-evidence.json').is_file():
    print('Restored dataset/completed v3 available; no new learner collection needed.')
else:
    runtime.start(V2)
    learner_reports = collect(COLLECTION_DIR, model_info=runtime.active_model_info)
    print('Learner trajectories saved:', COLLECTION_DIR)

## Verify teacher recoveries and build the mixture — prework

Select up to 24 learner-visited roots per game: fatal choices with safe alternatives, earlier same-life decisions leading into traps, pellet stalls, expiring power, respawn and maze cleanup. Replay the exact learner prefix, then let the frozen qualified teacher continue until clear, game-over or watchdog. A recovery supplies labels only when it clears the maze and passes the existing safety/loop/stall gates. Failed attempts remain visible in `teacher-recoveries.json` and the native replay bundle. An immediately safe retreat alone is not evidence of a complete recovery.

Training mixes **2,048 unchanged canonical v2 expert examples + 2,048 newly verified corrections/recovery examples**, plus the trainer's **2,000 generic decision-v7 replay requests**. Fresh development has 512 labels and never enters training. Coverage floors include at least 128 learner-visited roots, 64 learner/teacher disagreements, 512 immediate safe/fatal decisions and 256 critical reverse labels in the new training half; development requires at least 64 critical decisions and 32 critical reversals. These are admission requirements, not claims that collection has already met them. Insufficient coverage stops the pipeline and reports what is missing.

Full native replays independently verify teacher recoveries before the dataset receipt is published. Search plans, future diagnostics and teacher targets remain outside Kev's inference input. The loss remains the published option softmax cross-entropy; all legal competing actions, including dangerous ones, remain options.

The native JavaScript teacher runs on CPU. `TEACHER_WORKERS=None` automatically uses available physical cores, respecting CPU affinity and cgroup quotas; set a positive integer to override. A VM with 48 available logical CPUs / 24 physical cores selects 24 workers. The cell prints its allocation and a generation-only ETA after the first worker wave; recovery lengths vary, so this is an estimate. Interrupt stops submissions and lets active workers close after their current native query. Rerunning reuses completed attempt receipts and retries incomplete attempts. This interrupt behavior applies to this updated helper; a cell already running an older helper keeps its old pool.

With Drive enabled, teacher prework backs up at generation start, every five minutes, and on orderly interruption/completion. Incremental verified ZIPs contain the learner inputs and completed accepted/rejected teacher attempts under `MyDrive/QPlusLearning/lab-01-kev-pacman/prework-backups/v3-collection/`; active attempts are retried. Wait for **Teacher Drive backup complete**. A failed periodic backup stops generation while retaining local progress. After a runtime replacement, run setup, Drive storage and the adapter-loading cell to restore an absent collection, then rerun this generation cell. Backups are checked before restore and existing local collections are preserved. A hard runtime loss can lose work since the last acknowledged backup. The older running helper needs to be reloaded to enable this automatic schedule.

In [ ]:
# Offline teacher corrections, full recovery proof and balanced dataset
runtime.stop()  # GPU is not needed for the frozen CPU teacher
TEACHER_WORKERS = None  # Automatic physical-core allocation; use 24 for an explicit override
if not (V3 / 'run-evidence.json').is_file():
    v3_manifest = build(COLLECTION_DIR, V3_DATA, QUALIFICATION, workers=TEACHER_WORKERS,
                        backup_directory=TEACHER_BACKUP_ROOT)
    assert v3_manifest['learner']['checkpoint_sha256'] == checkpoint_fingerprint(V2), 'Dataset used another v2 adapter'
    print('Accepted teacher recoveries:', sum(a['accepted'] for a in v3_manifest['attempts']))
    print('Rejected attempts retained:', sum(not a['accepted'] for a in v3_manifest['attempts']))
    print('V3 dataset:', json.dumps({k: v3_manifest[k] for k in ['counts', 'coverage', 'expected_optimizer_steps']}, indent=2))
else:
    print('Completed v3 restored; dataset generation is not required for playback/evaluation.')

## Train native-v3 from v2 weights

One epoch, learning rate 2e-5, batch 4 × accumulation 2, BF16 autocast, FP32 stored weights, checkpointing and required optimized kernels. The full mixture stays **6,096 requests / 762 optimizer updates**. A fresh v3 stage loads v2 LoRA/head weights and creates a new optimizer/scheduler. If v3 is interrupted, rerunning this cell resumes the latest **v3** recovery with matching inputs; it never uses v2's completed optimizer progress.

Recovery/Drive saves occur at step 1, every 100 steps or five minutes, and completion. Watch printed save/backup acknowledgments and TensorBoard. Full training duration and improvement need an actual GPU run.

In [ ]:
# Warm-start a new v3 stage; only interrupted v3 training resumes
if (V3 / 'run-evidence.json').is_file():
    print('Using completed v3 checkpoint:', V3)
else:
    v3_manifest = validate_dataset(V3_DATA, QUALIFICATION)
    assert v3_manifest['learner']['checkpoint_sha256'] == checkpoint_fingerprint(V2), 'Dataset was collected with different v2 weights'
    RESUME_V3 = latest_snapshot(Path(str(V3) + '-recovery')) is not None
    print('Resume interrupted v3:', RESUME_V3)
    print('Selected v3 command:', runtime.finetuning_command(V3_TRAINING, V3, V2))
    runtime.finetune(V3_TRAINING, V3, init_from=V2, steps=0, resume=RESUME_V3)
    metrics = json.loads((V3 / 'training_metrics.json').read_text())
    assert metrics['optimizer_steps'] == 762 and metrics['records_seen'] == metrics['requested_records'] == 6096
    assert not metrics.get('truncated_records', 0) and not metrics.get('rejected_records', 0)
    evidence = json.loads((V3 / 'run-evidence.json').read_text())
    assert evidence['parent_checkpoint_sha256'] == checkpoint_fingerprint(V2)
print('V3 training metrics:', (V3 / 'training_metrics.json').read_text())

## Compare full native games: v2 versus v3

Use the unchanged 20 reserved starts and v2 observation/action protocol. Continue after each death until maze clear or native game-over. Save every trajectory and report clears, remaining food, avoidable deaths, conditional fatal-choice rate, pellet-free cycles/stalls, power use and per-life progress. No planner intervenes. Scores alone do not select the better checkpoint. Teacher qualification remains separate from model results.

In [ ]:
# Paired native-v2 versus native-v3 evaluation; these states never enter training
comparison_run = new_run_directory(LAB_DIR / 'results', 'comparison-v2-v3',
                                   baseline_checkpoint=str(V2), fine_tuned_checkpoint=str(V3), action_version=2)
runtime.start(V2)
before_gameplay = benchmark_gameplay(model_info=runtime.active_model_info, trace_dir=comparison_run / 'native-v2')
(comparison_run / 'native-v2.json').write_text(json.dumps(before_gameplay, indent=2))
runtime.start(V3)
after_gameplay = benchmark_gameplay(model_info=runtime.active_model_info, trace_dir=comparison_run / 'native-v3')
(comparison_run / 'native-v3.json').write_text(json.dumps(after_gameplay, indent=2))
comparison = {'schema_version': 3, 'baseline_checkpoint': str(V2), 'fine_tuned_checkpoint': str(V3),
              'gameplay': {'general': before_gameplay, 'fine_tuned': after_gameplay,
                          'paired': paired_gameplay(before_gameplay, after_gameplay)},
              'behavior': {'native_v2': behavior_summary(before_gameplay), 'native_v3': behavior_summary(after_gameplay)},
              'recording_directory': str(comparison_run)}
(comparison_run / 'comparison.json').write_text(json.dumps(comparison, indent=2))
(LAB_DIR / 'comparison-v2-v3.json').write_text(json.dumps(comparison, indent=2))
for name, games in [('native-v2', before_gameplay), ('native-v3', after_gameplay)]:
    print(name, 'clears:', games['level_clears'], 'means:', games['means'], 'incomplete:', games['capped_episodes'])
print('Retreat, safety and per-life progress:', json.dumps(comparison['behavior'], indent=2))
print('All benchmark trajectories:', comparison_run)

## Interactive play with native-v3 — ungraded

Choose Kev and Start. Check the visible checkpoint badge says `kev-4b-pacman-native-v3`; it displays the active LoRA/head hashes. The maze, four ghost mechanisms and native action boundaries match the benchmark. Every invocation records to its own dated directory. Pause the board before benchmarking or backup.

In [ ]:
# Interactive play with the completed Pac-Man adapter
from IPython.display import display, HTML, JSON
PACMAN_PLAY_CHECKPOINT = CHECKPOINT_ROOT / 'kev-4b-pacman-native-v3'
# To inspect your existing v1 adapter, explicitly select CHECKPOINT_ROOT / 'kev-4b-pacman-planner-v1' instead.
if not (PACMAN_PLAY_CHECKPOINT / 'run-evidence.json').is_file():
    raise RuntimeError(f'Complete Stage 5 or restore its completed checkpoint to {PACMAN_PLAY_CHECKPOINT} before playing.')
if runtime.active_checkpoint is None or runtime.active_model_info()['checkpoint'] != str(PACMAN_PLAY_CHECKPOINT):
    runtime.start(PACMAN_PLAY_CHECKPOINT)
if not runtime.active_model_info()['pacman_fine_tuned']:
    raise RuntimeError('The active checkpoint is not recorded as Pac-Man fine-tuned. Load the completed Stage 5 checkpoint.')
fine_tuned_play_run = new_run_directory(LAB_DIR / 'results', 'interactive-fine-tuned',
                                       active_checkpoint=runtime.active_model_info(), seed=7, action_version=2)
fine_tuned_bridge = NotebookBridge(fine_tuned_play_run / 'fine-tuned-player-trace.jsonl', runtime.active_model_info)
print('Interactive state/action/probability trace:', fine_tuned_bridge.trace_path)
try:
    from google.colab import output
except ImportError:
    print('Kaggle/local: use the Python rollouts below; interactive callbacks require Colab.')
else:
    output.register_callback('pacman.decide', lambda state, selected_bridge=fine_tuned_bridge: JSON(selected_bridge.decide(state)))
    output.register_callback('pacman.model', lambda selected_bridge=fine_tuned_bridge: JSON(selected_bridge.model()))
    display(HTML(GAME))

## Back up v2/v3 checkpoints and all evidence to Google Drive

Run this before replacing the runtime, including after data prework. Checkpoint backups remain separate for each adapter. V3 checkpoint recovery also restores its exact input, dataset receipt, development split and native replay evidence. The dated session archive retains all learner trajectories and teacher attempts, including failures, every benchmark episode, interactive trace and training curve. Wait for `Session backup complete` and an empty `checkpoint_errors` list.

In [ ]:
# Back up all available checkpoints, trajectories and traces to Google Drive
from pathlib import Path
import json
from google.colab import drive
from checkpoint_backup import backup_lab_to_drive
drive.mount('/content/drive')
runtime.stop()
DRIVE_LAB_ROOT = Path('/content/drive/MyDrive/QPlusLearning/lab-01-kev-pacman')
drive_receipt = backup_lab_to_drive(LAB_DIR, DRIVE_LAB_ROOT, checkpoint_root=CHECKPOINT_ROOT)
print(json.dumps(drive_receipt, indent=2))
if drive_receipt['checkpoint_errors']:
    raise RuntimeError('Traces were saved, but some checkpoint backups failed. See checkpoint_errors above and retry.')